# Fourier analysis of real and AI-generated images

Notebook này chuẩn hóa phép so sánh Fourier để tránh các kết luận sai do kích thước ảnh, DC component, discontinuity ở biên và auto-scaling của `imshow`.

Quy trình cho mỗi ảnh:

```text
center crop → resize cùng kích thước → luminance → subtract mean
→ 2D Hann window → FFT → normalized power spectral density → log10
```

Figure định tính dùng **chung một color scale**. Phần cuối tổng hợp **radial PSD trên nhiều ảnh** theo từng generator; đây mới là kết quả nên dùng để hình thành giả thuyết về Fourier artifact.


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT = Path("/kaggle/input/datasets/yangsangtai/tiny-genimage")

RANDOM_SEED = 42
IMAGE_SIZE = 256
NUM_VISUAL_PER_GENERATOR = 1
MAX_STATS_PER_GROUP = 100
RADIAL_BINS = 96

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
rng = random.Random(RANDOM_SEED)


def get_image_paths(folder: Path) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def is_generator_dir(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "val" / "nature").exists()
        and (path / "val" / "ai").exists()
    )


# Không lọc bằng prefix "imagenet_ai_" vì cách đó bỏ sót GLIDE và Midjourney.
generator_dirs = sorted(
    (path for path in DATA_ROOT.iterdir() if is_generator_dir(path)),
    key=lambda path: path.name,
)

print("Generators found:", len(generator_dirs))
for generator_dir in generator_dirs:
    real_count = len(get_image_paths(generator_dir / "val" / "nature"))
    fake_count = len(get_image_paths(generator_dir / "val" / "ai"))
    print(f"{generator_dir.name}: real={real_count}, fake={fake_count}")


In [ ]:
def sample_paths(paths: list[Path], count: int) -> list[Path]:
    if len(paths) <= count:
        return list(paths)
    return rng.sample(paths, count)


# Cân bằng figure: mỗi generator đóng góp cùng số ảnh real và fake.
visual_pairs = []
for generator_dir in generator_dirs:
    real_paths = get_image_paths(generator_dir / "val" / "nature")
    fake_paths = get_image_paths(generator_dir / "val" / "ai")
    pair_count = min(NUM_VISUAL_PER_GENERATOR, len(real_paths), len(fake_paths))

    selected_real = sample_paths(real_paths, pair_count)
    selected_fake = sample_paths(fake_paths, pair_count)
    for real_path, fake_path in zip(selected_real, selected_fake):
        visual_pairs.append(
            {
                "source": generator_dir.name,
                "real_path": real_path,
                "fake_path": fake_path,
            }
        )

print("Balanced visual pairs:", len(visual_pairs))


In [ ]:
def center_crop_square(image: Image.Image) -> Image.Image:
    width, height = image.size
    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    return image.crop((left, top, left + side, top + side))


def load_standardized_image(image_path: Path, image_size: int = IMAGE_SIZE):
    image = Image.open(image_path).convert("RGB")
    original_size = image.size
    image = center_crop_square(image)
    image = image.resize((image_size, image_size), Image.Resampling.LANCZOS)

    rgb = np.asarray(image, dtype=np.float32) / 255.0
    # Luminance trên ảnh sRGB. Phân tích channel/color-space riêng nên làm ở notebook khác.
    gray = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    return rgb, gray.astype(np.float32), original_size


def compute_normalized_psd(gray_image: np.ndarray):
    height, width = gray_image.shape

    # Loại DC component để độ sáng trung bình không thống trị tâm phổ.
    centered = gray_image - float(gray_image.mean())

    # Giảm spectral leakage do DFT giả định ảnh lặp tuần hoàn.
    window = np.outer(np.hanning(height), np.hanning(width)).astype(np.float32)
    windowed = centered * window

    spectrum = np.fft.fftshift(np.fft.fft2(windowed))
    psd = np.abs(spectrum) ** 2

    # Chuẩn hóa tổng năng lượng để so sánh hình dạng phổ thay vì contrast tổng thể.
    psd = psd / (psd.sum() + np.finfo(np.float64).eps)
    log_psd = np.log10(psd + 1e-12)
    return psd, log_psd


def radial_psd(psd: np.ndarray, bins: int = RADIAL_BINS):
    height, width = psd.shape
    yy, xx = np.indices((height, width))
    cy, cx = (height - 1) / 2.0, (width - 1) / 2.0

    # r=1 tương ứng bán kính Nyquist theo trục ngắn; bỏ các góc ngoài r>1.
    radius = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2) / (min(height, width) / 2.0)
    edges = np.linspace(0.0, 1.0, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    profile = np.full(bins, np.nan, dtype=np.float64)

    for index in range(bins):
        mask = (radius >= edges[index]) & (radius < edges[index + 1])
        if np.any(mask):
            profile[index] = float(psd[mask].mean())

    # Chuẩn hóa area dưới radial curve để so sánh shape giữa các ảnh.
    profile = profile / (np.nansum(profile) + np.finfo(np.float64).eps)
    return centers, profile


In [ ]:
# Tính trước toàn bộ spectrum để dùng chung một thang màu.
visual_rows = []
all_log_psd = []

for pair in visual_pairs:
    real_rgb, real_gray, real_original_size = load_standardized_image(pair["real_path"])
    fake_rgb, fake_gray, fake_original_size = load_standardized_image(pair["fake_path"])
    _, real_log_psd = compute_normalized_psd(real_gray)
    _, fake_log_psd = compute_normalized_psd(fake_gray)

    visual_rows.append(
        {
            **pair,
            "real_rgb": real_rgb,
            "fake_rgb": fake_rgb,
            "real_log_psd": real_log_psd,
            "fake_log_psd": fake_log_psd,
            "real_original_size": real_original_size,
            "fake_original_size": fake_original_size,
        }
    )
    all_log_psd.extend([real_log_psd, fake_log_psd])

stacked_log_psd = np.stack(all_log_psd)
vmin, vmax = np.percentile(stacked_log_psd, [2.0, 99.5])

fig, axes = plt.subplots(
    len(visual_rows),
    4,
    figsize=(16, max(4, len(visual_rows) * 3.6)),
    squeeze=False,
)

for row_index, row in enumerate(visual_rows):
    source = row["source"]

    axes[row_index, 0].imshow(row["real_rgb"])
    axes[row_index, 0].set_title(f"REAL — {source}\noriginal={row['real_original_size']}")

    axes[row_index, 1].imshow(
        row["real_log_psd"], cmap="magma", vmin=vmin, vmax=vmax
    )
    axes[row_index, 1].set_title("REAL normalized log PSD")

    axes[row_index, 2].imshow(row["fake_rgb"])
    axes[row_index, 2].set_title(f"FAKE — {source}\noriginal={row['fake_original_size']}")

    axes[row_index, 3].imshow(
        row["fake_log_psd"], cmap="magma", vmin=vmin, vmax=vmax
    )
    axes[row_index, 3].set_title("FAKE normalized log PSD")

    for column_index in range(4):
        axes[row_index, column_index].axis("off")

colorbar = fig.colorbar(
    axes[0, 3].images[0],
    ax=axes[:, 1::2].ravel().tolist(),
    fraction=0.015,
    pad=0.02,
)
colorbar.set_label("log10 normalized power")
fig.suptitle(
    "Controlled Fourier comparison: same size, mean removed, Hann window, shared scale",
    y=1.001,
)
plt.tight_layout()
plt.show()


In [ ]:
def summarize_profile(profile: np.ndarray, radii: np.ndarray) -> dict:
    valid = np.isfinite(profile) & (profile > 0)
    normalized = profile[valid] / profile[valid].sum()
    entropy = -float(np.sum(normalized * np.log(normalized + 1e-12)))
    high_frequency_ratio = float(np.nansum(profile[radii >= 0.5]))
    return {
        "high_frequency_ratio": high_frequency_ratio,
        "spectral_entropy": entropy,
    }


profile_rows = []

for generator_dir in generator_dirs:
    source = generator_dir.name
    for label_name, folder_name in [("real", "nature"), ("fake", "ai")]:
        paths = get_image_paths(generator_dir / "val" / folder_name)
        selected_paths = sample_paths(paths, min(MAX_STATS_PER_GROUP, len(paths)))

        for image_path in selected_paths:
            _, gray, original_size = load_standardized_image(image_path)
            psd, _ = compute_normalized_psd(gray)
            radii, profile = radial_psd(psd)
            metrics = summarize_profile(profile, radii)
            profile_rows.append(
                {
                    "source": source,
                    "label": label_name,
                    "path": str(image_path),
                    "original_width": original_size[0],
                    "original_height": original_size[1],
                    "radii": radii,
                    "profile": profile,
                    **metrics,
                }
            )

print("Images analyzed for radial PSD:", len(profile_rows))

fig, axes = plt.subplots(
    len(generator_dirs),
    1,
    figsize=(10, max(4, 3.2 * len(generator_dirs))),
    sharex=True,
    squeeze=False,
)

colors = {"real": "tab:blue", "fake": "tab:orange"}

for axis, generator_dir in zip(axes[:, 0], generator_dirs):
    source = generator_dir.name
    for label_name in ["real", "fake"]:
        selected = [
            row["profile"]
            for row in profile_rows
            if row["source"] == source and row["label"] == label_name
        ]
        profiles = np.stack(selected)
        median = np.nanmedian(profiles, axis=0)
        q25 = np.nanpercentile(profiles, 25, axis=0)
        q75 = np.nanpercentile(profiles, 75, axis=0)

        axis.plot(radii, median, label=label_name.upper(), color=colors[label_name])
        axis.fill_between(radii, q25, q75, color=colors[label_name], alpha=0.18)

    axis.set_yscale("log")
    axis.set_ylabel("Radial PSD")
    axis.set_title(source)
    axis.grid(alpha=0.2)
    axis.legend()

axes[-1, 0].set_xlabel("Normalized spatial frequency (0=center, 1=axis Nyquist)")
fig.suptitle("Median radial PSD with interquartile range", y=1.002)
plt.tight_layout()
plt.show()

metric_df = pd.DataFrame(
    {
        key: value
        for key, value in row.items()
        if key not in {"radii", "profile"}
    }
    for row in profile_rows
)

summary_df = (
    metric_df.groupby(["source", "label"])[
        ["high_frequency_ratio", "spectral_entropy"]
    ]
    .agg(["count", "median", "mean", "std"])
    .round(6)
)

display(summary_df)


## Cách đọc kết quả

- Không kết luận từ một cross hoặc một tia sáng riêng lẻ: chúng có thể đến từ cạnh và nội dung ảnh.
- Chỉ xem một dấu hiệu là ứng viên forensic khi chênh lệch real–fake lặp lại trên nhiều ảnh của cùng generator, tồn tại ở generator chưa thấy, và còn ổn định sau JPEG/resize.
- Radial PSD làm mất thông tin hướng. Nếu radial curve khác nhau rõ rệt, bước tiếp theo là bổ sung angular spectrum, periodic-peak score và FFT của noise residual/SRM/NPR.
